# Conversational App for Itinerary Planning

## Sample bot with no memory

In [6]:

from dotenv import load_dotenv
# 2. Modern Import: core schema for messages
from langchain_core.messages import HumanMessage, SystemMessage
# 1. Modern Import: dedicated OpenAI package
from langchain_openai import ChatOpenAI

load_dotenv()

# Setup the Model
# recommended: specify a model (e.g., gpt-4o, gpt-4o-mini, gpt-3.5-turbo)
chat = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# Create the message list
messages = [
    SystemMessage(content="You are a helpful assistant that help the user to plan an optimized itinerary."),
    HumanMessage(content="I'm going to Frankfurt for 2 days, what can I visit?")
]

# Invoke the model
# .invoke() returns an AIMessage object
output = chat.invoke(messages)

# Print the content
print(output.content)


Frankfurt is a vibrant city with a mix of modern skyscrapers and historic sites. Here’s a suggested two-day itinerary that covers key attractions, cultural experiences, and some local cuisine:

### Day 1: Explore the City Center and Historical Sites

**Morning:**
1. **Römer** - Start your day at the Römer, the iconic medieval building that serves as Frankfurt's city hall. Take some time to explore the surrounding Römerberg square and its picturesque half-timbered houses.
2. **St. Bartholomew's Cathedral (Dom Sankt Bartholomeus)** - Just a short walk away, this impressive Gothic cathedral is worth a visit. Climb the tower for a panoramic view of the city.

**Lunch:**
- Grab a traditional Frankfurt meal at a nearby restaurant. Try "Handkäse mit Musik" (a local cheese specialty) or "Frankfurter Würstchen" (sausages).

**Afternoon:**
3. **Palmengarten** - Spend the afternoon in this beautiful botanical garden. It’s a serene place to relax and enjoy nature.
4. **Senckenberg Museum of Natura

## Adding Memory

In [7]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import ConversationChain

memory = ConversationBufferMemory()
conversation = ConversationChain(
    llm=chat, verbose=True, memory=memory
)

conversation.run("Hi there!")

/var/folders/dw/6h6kf7c50qq6d4ndkybdm4980000gn/T/ipykernel_24935/2165110715.py:4: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory()
/var/folders/dw/6h6kf7c50qq6d4ndkybdm4980000gn/T/ipykernel_24935/2165110715.py:5: LangChainDeprecationWarning: The class `ConversationChain` was deprecated in LangChain 0.2.7 and will be removed in 1.0. Use `langchain_core.runnables.history.RunnableWithMessageHistory` instead.
  conversation = ConversationChain(
/var/folders/dw/6h6kf7c50qq6d4ndkybdm4980000gn/T/ipykernel_24935/2165110715.py:9: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  conversation.run("Hi there!")




> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:

Human: Hi there!
AI:

> Finished chain.


"Hello! How are you doing today? I'm here to chat about anything on your mind or help with any questions you might have. What's up?"

In [8]:
conversation.run("what is the most iconic place in Frankfurt ?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi there!
AI: Hello! How are you doing today? I'm here to chat about anything on your mind or help with any questions you might have. What's up?
Human: what is the most iconic place in Frankfurt ?
AI:

> Finished chain.


"One of the most iconic places in Frankfurt is the Römer, which is a historic building that has served as the city hall for over 600 years. It features a beautiful stepped gable design and is located in the heart of Frankfurt’s Altstadt (Old Town). The Römer is surrounded by picturesque medieval buildings and is a central part of Frankfurt’s history and culture. Another landmark is the Main Tower, which offers stunning panoramic views of the city from its observation deck. If you're interested in modern architecture, the skyline of Frankfurt is also quite famous, with its striking skyscrapers like the Deutsche Bank Twin Towers and the EZB (European Central Bank) building. Have you been to Frankfurt before?"

In [12]:
conversation.run("What kind of other events?")



> Entering new ConversationChain chain...
Prompt after formatting:
The following is a friendly conversation between a human and an AI. The AI is talkative and provides lots of specific details from its context. If the AI does not know the answer to a question, it truthfully says it does not know.

Current conversation:
Human: Hi there!
AI: Hello! How can I assist you today?
Human: what is the most iconic place in Rome?
AI: The most iconic place in Rome is probably the Colosseum. It is a magnificent amphitheater that was built in the first century AD and is one of the most recognizable symbols of ancient Rome. The Colosseum was used for gladiatorial contests, public spectacles, and other events. Today, it is a major tourist attraction and a UNESCO World Heritage site.
Human: What kind of other events?
AI:

> Finished chain.


'Other events that took place at the Colosseum include mock sea battles, animal hunts, and reenactments of famous battles. The Colosseum was also used for executions and religious ceremonies. It was a versatile venue that could accommodate a variety of events and entertainments.'

In [13]:
memory.load_memory_variables({})

{'history': 'Human: Hi there!\nAI: Hello! How can I assist you today?\nHuman: what is the most iconic place in Rome?\nAI: The most iconic place in Rome is probably the Colosseum. It is a magnificent amphitheater that was built in the first century AD and is one of the most recognizable symbols of ancient Rome. The Colosseum was used for gladiatorial contests, public spectacles, and other events. Today, it is a major tourist attraction and a UNESCO World Heritage site.\nHuman: What kind of other events?\nAI: Other events that took place at the Colosseum include mock sea battles, animal hunts, and reenactments of famous battles. The Colosseum was also used for executions and religious ceremonies. It was a versatile venue that could accommodate a variety of events and entertainments.'}

In [4]:

from dotenv import load_dotenv

# 1. Modern Imports
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# 2. LangGraph Imports (The modern "Memory" architecture)
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, MessagesState, StateGraph

load_dotenv()

# --- Setup Model ---
chat = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# --- Define the Graph (The "ConversationChain" replacement) ---

# We use 'MessagesState' which natively handles a list of messages (Memory)
workflow = StateGraph(state_schema=MessagesState)

def call_model(state: MessagesState):
    """
    The node that calls the LLM.
    It receives the full history (state['messages']) automatically.
    """
    response = chat.invoke(state["messages"])
    # We return the new message, which LangGraph appends to the history
    return {"messages": response}

# Define the flow: Start -> Model -> End
workflow.add_edge(START, "model")
workflow.add_node("model", call_model)

# --- Setup Persistence (The "ConversationBufferMemory" replacement) ---
# MemorySaver keeps the state in RAM (like ConversationBufferMemory)
memory = MemorySaver()

# Compile the graph with memory
app = workflow.compile(checkpointer=memory)

# --- Usage ---

# 1. Define a Thread ID (This isolates the user's session)
config = {"configurable": {"thread_id": "session_1"}}

# 2. First Interaction
input_1 = "Hi there! I want to plan a trip to Frankfurt in the summer."
print(f"User: {input_1}")

# We pass the input message.
# Note: We can also pass a SystemMessage here if we want to set behavior.
response_1 = app.invoke(
    {"messages": [HumanMessage(content=input_1)]},
    config=config
)
print(f"Bot:  {response_1['messages'][-1].content}")

# 3. Second Interaction (Proving Memory Works)
input_2 = "what is the most iconic place in the place I want to visit ?"
print(f"\nUser: {input_2}")

response_2 = app.invoke(
    {"messages": [HumanMessage(content=input_2)]},
    config=config
)
print(f"Bot:  {response_2['messages'][-1].content}")

print("\n\n--- History ---\n")

# history object
history = app.get_state_history(config=config)

# Iterate and print the state values at each checkpoint
for i, snapshot in enumerate(history):
    print(f"--- Checkpoint {i} ---")
    print(snapshot.values)


User: Hi there! I want to plan a trip to Frankfurt in the summer.
Bot:  That sounds like a great idea! Frankfurt is a vibrant city with a mix of modern and historic attractions. Here are some tips and suggestions to help you plan your trip:

### Best Time to Visit
- **Summer (June to August)**: This is a popular time to visit due to warm weather and numerous festivals. Keep in mind that it can be quite busy, especially in July and August.

### Things to Do
1. **Römer**: Explore the historic town hall of Frankfurt, which dates back to the 15th century. The surrounding Römerberg square is picturesque and often hosts events.
   
2. **Frankfurt Cathedral (Kaiserdom)**: Visit this stunning Gothic cathedral, which is one of the tallest churches in Germany and offers great views from the tower.

3. **Palmengarten**: This beautiful botanical garden is perfect for a leisurely stroll and offers a range of exotic plants and themed gardens.

4. **Museumsufer (Museum Embankment)**: This area along 

## Adding non parametric knowledge

In [1]:
import os
from dotenv import load_dotenv

# 1. Modern Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# 2. Modern Chain Imports
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv()

# --- Setup Data (Same logic, modern classes) ---
try:
    # Ensure file exists or create dummy for testing
    if not os.path.exists('Travel_Italy.pdf'):
        print("Note: 'Travel_Italy.pdf' not found. Code will fail at loader step.")

    loader = PyPDFLoader('Travel_Italy.pdf')
    raw_documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
    documents = text_splitter.split_documents(raw_documents)

    embeddings = OpenAIEmbeddings()
    db = FAISS.from_documents(documents, embeddings)
    retriever = db.as_retriever()

    print("Vector Store created successfully.")
except Exception as e:
    print(f"Error setting up data: {e}")
    exit()

# --- Modern Conversational RAG Implementation ---

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 1. Contextualize Question Sub-chain
# This replaces the internal logic of ConversationalRetrievalChain that rephrased the question.
# It defines: "If there is history, rephrase the new question to be standalone."
contextualize_q_system_prompt = (
    "Given a chat history and the latest user question "
    "which might reference context in the chat history, "
    "formulate a standalone question which can be understood "
    "without the chat history. Do NOT answer the question, "
    "just reformulate it if needed and otherwise return it as is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", contextualize_q_system_prompt),
        # This placeholder will be replaced by the actual list of messages
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

# This chain takes history + input -> generates a search query
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)

# 2. Answer Question Sub-chain
# This is the standard "Stuff" chain that answers based on retrieved docs.
qa_system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# 3. Final Retrieval Chain
# This combines the "History Aware" retrieval with the "Question Answering"
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

# --- Usage (Managing Memory) ---

# Instead of ConversationBufferMemory object, we simply manage a list of messages.
# This gives you full control over storage (Redis, SQL, etc) in production.
chat_history = []

query = "Give me some review about the Pantheon"

# Invoke
result = rag_chain.invoke({
    "input": query,
    "chat_history": chat_history
})

print(f"User: {query}")
print(f"AI: {result['answer']}")

# Update History manually (Modern Pattern)
chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=result["answer"])
])

# Test Follow-up (to prove memory works)
query_2 = "Where is it located?" # "It" refers to Pantheon
result_2 = rag_chain.invoke({
    "input": query_2,
    "chat_history": chat_history
})

print(f"\nUser: {query_2}")
print(f"AI: {result_2['answer']}")

Vector Store created successfully.
User: Give me some review about the Pantheon
AI: The Pantheon is praised for its "angelic and non-human design," featuring a gigantic dome and harmonious architecture. Visitors appreciate its historical significance as one of the best-preserved ancient Roman monuments and its transformation into a church in 608 AD. The entrance portico with stout columns is particularly admired, making it a must-see in Rome.

User: Where is it located?
AI: The Pantheon is located at Piazza della Rotonda, Rome.


In [4]:
import os
from dotenv import load_dotenv

# 1. Modern Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# 2. Modern Chain Construction
from langchain_classic.chains import create_history_aware_retriever, create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

load_dotenv()

# --- Setup Data (Same as before) ---
try:
    if not os.path.exists('Travel_Frankfurt.pdf'):
        print("Warning: PDF not found. Using dummy data for demonstration.")
        with open("Travel_Frankfurt.pdf", "w") as f: f.write("Frankfurt has the Römer and the Main Tower.")

    loader = PyPDFLoader('Travel_Frankfurt.pdf')
    raw_documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
    documents = text_splitter.split_documents(raw_documents)

    embeddings = OpenAIEmbeddings()
    db = FAISS.from_documents(documents, embeddings)
    retriever = db.as_retriever()
except Exception as e:
    print(f"Error initializing data: {e}")
    exit()

# --- Modern RAG Construction ---

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 1. History-Aware Retriever (Replaces 'condense_question_prompt')
# This step takes the chat history and the new question, and rephrases the question
# so it makes sense to the vector database (e.g., resolving "it" or "there").
rephrase_system_prompt = (
    "Given the following conversation and a follow up question, "
    "rephrase the follow up question to be a standalone question."
)

rephrase_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", rephrase_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

history_aware_retriever = create_history_aware_retriever(
    llm, retriever, rephrase_prompt
)

# 2. Answer Generator (Replaces the internal QA logic)
# This is where we put your instruction to "ignore documents if answer not found".
qa_system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer the question. "
    "IMPORTANT: If you cannot find the answer in the document provided, "
    "ignore the document and answer based on your general knowledge anyway."
    "\n\n"
    "{context}"
)

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", qa_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ]
)

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)

# 3. Final Chain (Combines Retrieval + Answering)
rag_chain = create_retrieval_chain(history_aware_retriever, question_answer_chain)

# --- Usage (Replacing ConversationBufferMemory) ---

# In modern LangChain, we pass the history explicitly as a list of messages.
# This makes it easier to store in databases or pass between APIs.
chat_history = []

query = "What can I visit in Frankfurt ?"

print(f"User: {query}")

response = rag_chain.invoke({
    "input": query,
    "chat_history": chat_history
})

print(f"AI: {response['answer']}")

# Example: How to update history for the next turn
chat_history.extend([
    HumanMessage(content=query),
    AIMessage(content=response["answer"])
])

# Example: Follow up to prove history works
query_2 = "How high is Main Tower ?" # Refers to Main Tower if found
print(f"User: {query_2}")

response = rag_chain.invoke({
    "input": query_2,
    "chat_history": chat_history
})

print(f"AI: {response['answer']}")

# Example: How to update history for the next turn
chat_history.extend([
    HumanMessage(content=query_2),
    AIMessage(content=response["answer"])
])

print("----> history\n")
print(chat_history)


User: What can I visit in Frankfurt ?
AI: In Frankfurt, you can visit several notable attractions, including:

1. **Cathedral (Dom)** - A prominent landmark with a 95 m high tower, where ten emperors were crowned.
2. **Kaiserpfalz franconofurd (Archaeological Garden)** - Offers insights into the city's history with remnants from Roman times and the Carolingian royal court.
3. **Canvas House (Leinwandhaus)** - The oldest textile shop in Frankfurt, now housing the Museum of Comical Art.
4. **Iron Bridge** - A historic bridge offering scenic views.
5. **Customs Tower** - A late Gothic gate tower that is part of the Historical Museum.
6. **Wertheim House** - A historic building with architectural significance.
7. **Historical Museum** - Showcases the history and culture of Frankfurt.
8. **Old Nikolai Church** - A historic church with architectural interest.
9. **Römerberg** - The historic square in the heart of Frankfurt, featuring the Fountain of Justice.
10. **Fountain of Justice** - A p

In [4]:
import os
from dotenv import load_dotenv

# 1. Modern Data Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.tools.retriever import create_retriever_tool

# 2. LangGraph Imports
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver

load_dotenv()

# --- Setup Data ---
try:
    if not os.path.exists('Travel_Italy.pdf'):
        print("Note: PDF not found. Creating dummy data for demonstration.")
        with open("Travel_Italy.pdf", "w") as f: f.write("The Pantheon in Rome is a former Roman temple.")

    loader = PyPDFLoader('Travel_Italy.pdf')
    raw_documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
    documents = text_splitter.split_documents(raw_documents)

    embeddings = OpenAIEmbeddings()
    db = FAISS.from_documents(documents, embeddings)
    print("Vector Store created.")
except Exception as e:
    print(f"Error setting up data: {e}")
    exit()

# --- Setup Tools & Agent ---
tool = create_retriever_tool(
    db.as_retriever(),
    "italy_travel",
    "Searches and returns documents regarding Italy."
)
tools = [tool]

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
memory = MemorySaver()

# Create the Graph (Agent)
agent_executor = create_agent(llm, tools, checkpointer=memory)

# --- Execution Function ---
def run_agent_conversation(input1: str, input2: str, input3: str):
    """
    Runs a 3-turn conversation using the provided string parameters.
    """
    # Unique session ID to persist memory for this specific run
    config = {"configurable": {"thread_id": "session_dynamic_inputs"}}

    print(f"\n--- Turn 1: {input1} ---")
    response_1 = agent_executor.invoke(
        {"messages": [("user", input1)]},
        config=config
    )
    print(f"Agent: {response_1['messages'][-1].content}")

    print(f"\n--- Turn 2: {input2} ---")
    response_2 = agent_executor.invoke(
        {"messages": [("user", input2)]},
        config=config
    )
    print(f"Agent: {response_2['messages'][-1].content}")

    print(f"\n--- Turn 3: {input3} ---")
    response_3 = agent_executor.invoke(
        {"messages": [("user", input3)]},
        config=config
    )
    print(f"Agent: {response_3['messages'][-1].content}")

# --- Main Execution ---
if __name__ == "__main__":
    # Define your inputs here (not hardcoded in the function logic)
    turn1_text = "hi, i'm Vale"
    turn2_text = "Tell me something about Pantheon"
    turn3_text = "what can I visit in Germany in 5 days ?"

    # Pass them as parameters
    run_agent_conversation(turn1_text, turn2_text, turn3_text)


Vector Store created.

--- Turn 1: hi, i'm Vale ---
Agent: Hi Vale! How can I assist you today?

--- Turn 2: Tell me something about Pantheon ---
Agent: The Pantheon is one of the best-preserved ancient Roman monuments and is renowned for its architectural brilliance. Originally built as a temple to the gods, it was later transformed into a church in 608 AD, making it the first classical building to serve this purpose. The Pantheon features a magnificent dome, which was an engineering marvel of its time, and it remains the largest unreinforced concrete dome in the world.

Michelangelo famously described the Pantheon as having an "angelic and non-human design," highlighting its grandeur and harmony. The entrance is marked by a portico with impressive columns, and the interior is equally stunning, with a large oculus at the top of the dome that allows natural light to illuminate the space.

Visitors often appreciate the sheer size and beauty of the Pantheon, as well as its historical sig

## Adding external tools

In [19]:
from langchain_classic import SerpAPIWrapper
from langchain_classic.tools import Tool

import os
from dotenv import load_dotenv

load_dotenv()

key = os.environ["SERPAPI_API_KEY"]

search = SerpAPIWrapper()

In [20]:
tools = [
    Tool.from_function(
        func=search.run,
        name="Search",
        description="useful for when you need to answer questions about current events"
    ),
    create_retriever_tool(
        db.as_retriever(), 
        "italy_travel",
        "Searches and returns documents regarding Italy."
    )
    ]

agent_executor = create_conversational_retrieval_agent(llm, tools, memory_key='chat_history', verbose=True)

In [34]:
memory

ConversationBufferMemory(chat_memory=ChatMessageHistory(messages=[HumanMessage(content='Give me some review about the Pantheon', additional_kwargs={}, example=False), AIMessage(content='Miskita:\n"Angelic and non-human design," was how Michelangelo described the Pantheon 14 centuries after its construction. The highlights are the gigantic dome, the upper eye, the sheer size of the place, and the harmony of the whole building. We visited with a Roman guide which is exactly how you should (or shouldn\'t) visit the city, especially since they talk too much and have lots and lots of history of each building. And so we learned things like how the interior dome was filled with sand during construction. Or not, because it really is impossible to be sure of anything when one doesn\'t know Italian and the guide has a very thick Roman accent. But we loved the place, especially the entrance portico with its stout columns. \n\nAlmudena:\nThe Pantheon is one of the best preserved ancient Roman monu

In [21]:
agent_executor({"input": "what can I visit in India in 3 days?"})



> Entering new AgentExecutor chain...
India is a vast and diverse country with numerous attractions to explore. While it may be challenging to cover all the highlights in just three days, here are some popular destinations that you can consider visiting:

1. Delhi: Start your trip in the capital city of India, Delhi. Spend a day exploring the historical sites such as the Red Fort, Jama Masjid, Qutub Minar, and Humayun's Tomb. Don't miss a visit to the bustling markets of Chandni Chowk and indulge in delicious street food.

2. Agra: From Delhi, take a day trip to Agra, home to the iconic Taj Mahal. Marvel at the beauty of this UNESCO World Heritage Site, known as one of the Seven Wonders of the World. Also, visit the Agra Fort, another architectural marvel.

3. Jaipur: Head to Jaipur, the capital of Rajasthan, also known as the Pink City. Spend a day exploring the magnificent Amber Fort, City Palace, Hawa Mahal (Palace of Winds), and Jantar Mantar. Don't forget to indulge in Rajasthan

{'input': 'what can I visit in India in 3 days?',
 'chat_history': [HumanMessage(content='what can I visit in India in 3 days?', additional_kwargs={}, example=False),
  AIMessage(content="India is a vast and diverse country with numerous attractions to explore. While it may be challenging to cover all the highlights in just three days, here are some popular destinations that you can consider visiting:\n\n1. Delhi: Start your trip in the capital city of India, Delhi. Spend a day exploring the historical sites such as the Red Fort, Jama Masjid, Qutub Minar, and Humayun's Tomb. Don't miss a visit to the bustling markets of Chandni Chowk and indulge in delicious street food.\n\n2. Agra: From Delhi, take a day trip to Agra, home to the iconic Taj Mahal. Marvel at the beauty of this UNESCO World Heritage Site, known as one of the Seven Wonders of the World. Also, visit the Agra Fort, another architectural marvel.\n\n3. Jaipur: Head to Jaipur, the capital of Rajasthan, also known as the Pink 

In [23]:
agent_executor({"input": "what is the current wheather in Delhi ?"})



> Entering new AgentExecutor chain...

Invoking: `Search` with `{'query': 'current weather in Delhi'}`


Current Weather · 95°F Mostly sunny · RealFeel® 105°. Very Hot. RealFeel Guide. Very Hot. 101° to 107°. Caution advised. Danger of dehydration, heat stroke, heat ...The current weather in Delhi is 95°F (35°C) with mostly sunny conditions. The RealFeel® temperature is 105°F (41°C), indicating that it feels very hot. Caution is advised as there is a danger of dehydration, heat stroke, and heat-related issues. It is important to stay hydrated and take necessary precautions if you are in Delhi or planning to visit.

> Finished chain.


{'input': 'what is the current wheather in Delhi ?',
 'chat_history': [HumanMessage(content='what can I visit in India in 3 days?', additional_kwargs={}, example=False),
  AIMessage(content="India is a vast and diverse country with numerous attractions to explore. While it may be challenging to cover all the highlights in just three days, here are some popular destinations that you can consider visiting:\n\n1. Delhi: Start your trip in the capital city of India, Delhi. Spend a day exploring the historical sites such as the Red Fort, Jama Masjid, Qutub Minar, and Humayun's Tomb. Don't miss a visit to the bustling markets of Chandni Chowk and indulge in delicious street food.\n\n2. Agra: From Delhi, take a day trip to Agra, home to the iconic Taj Mahal. Marvel at the beauty of this UNESCO World Heritage Site, known as one of the Seven Wonders of the World. Also, visit the Agra Fort, another architectural marvel.\n\n3. Jaipur: Head to Jaipur, the capital of Rajasthan, also known as the Pi

In [24]:
agent_executor({"input": "I'm travelling to Italy, can you give me some suggestions of the main attractions?"})



> Entering new AgentExecutor chain...

Invoking: `italy_travel` with `{'query': 'main attractions in Italy'}`


[Document(page_content='ITALY\nMINUBE TRAVEL GUIDE\nThe best must-see places for your travels, all\ndiscovered by real minube users. Enjoy!', metadata={'source': 'italy_travel.pdf', 'page': 0}), Document(page_content='What to see\n in Italy\nPage 10\n58\nGardens\nParco Sempione\n \nPatricia M. Gómez:\n \nThis is one of the most important\nparks in Milan. \nIt was wonderful to visit it this winter all\ncovered in snow and then this spring for our first "picnic". The\npark is very well kept and clean. \nYou can easily go to\nworkout, walk or just have a little taste of green space in this\ncity, which can sometimes be gray. \n☎\n 390 277 404 343\n - \nPiazza Castello, 20121 Milan, Italy, \nMilan\n59\nExhibitions\nGulf of Naples\n \nCarlos Millán Gómez:\n \nIs there anywhere in the world\nmore beautiful than the Gulf of Naples? I would say no. It is\nthe very center of Paradis

{'input': "I'm travelling to Italy, can you give me some suggestions of the main attractions?",
 'chat_history': [HumanMessage(content='what is the current wheather in India?', additional_kwargs={}, example=False),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'Search', 'arguments': '{\n  "query": "current weather in India"\n}'}}, example=False),
  FunctionMessage(content='Ahmedabad 89° Bengaluru 86° Bhopal 84° Chandigarh 91° Chennai 91° Delhi 97° Dispur 88° Faridabad 94° Guwahati 88° Jaipur 93° Kolkata 95° Lucknow 94° ...', additional_kwargs={}, name='Search'),
  AIMessage(content="I apologize, but I couldn't find the specific current weather information for India. The weather conditions can vary significantly across different regions of the country. It is recommended to check a reliable weather website or use a weather app to get the most up-to-date and accurate information about the weather in specific cities or regions of India.", additional_kwargs={}, exampl